# Sample — 100 suspended / 100 active accused accounts

A random sample of accounts accused in r/TheseFuckingAccounts, stratified by PRAW account outcome.

**Sampling frame:** accounts in `accused_accounts_not_deleted.parquet` whose Arctic Shift history pull
is marked `complete` in `arctic_pull_done.parquet`, so every sampled account has its posts and
comments on disk. Accounts with no archive history and the incomplete or truncated pulls are left out.

Nothing is re-pulled. Outputs go to `sample_200_output/`:
- `sample_accounts.parquet`: one row per account, with accusation info, Arctic Shift metadata and the PRAW profile (PRAW returns no profile fields for suspended accounts)
- `sample_posts.parquet` and `sample_comments.parquet`: the full stored history of the 200 accounts, deduplicated on `id`

In [ ]:
# Libraries, input/output folders, group size (100 per account status) and the random seed.
import pandas as pd
import numpy as np
import os
import re
import glob
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

DATA_DIR = os.path.join('..', 'p2_data_cleaning', 'data')
OUT_DIR = 'sample_200_output'
N_PER_GROUP = 100
SEED = 2026
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Sampling frame

In [ ]:
# Sampling frame: accused accounts that are not deleted, joined to the Arctic Shift pull log.
# Only accounts whose history pull finished (complete == True) can be sampled.
accused = pd.read_parquet(os.path.join(DATA_DIR, 'accused_accounts_not_deleted.parquet'))
pull_done = pd.read_parquet(os.path.join(DATA_DIR, 'arctic_pull_done.parquet'))

frame = accused.merge(pull_done, on='username_key', how='inner', validate='one_to_one')
print('accused, not deleted:')
print(accused['account_status'].value_counts().to_string())
print('\nwith any Arctic pull:')
print(frame['account_status'].value_counts().to_string())
frame = frame[frame['complete']]
print('\nwith a complete pull (sampling frame):')
print(frame['account_status'].value_counts().to_string())

## 2. Draw the sample

In [ ]:
# Draw 100 suspended and 100 active accounts at random (fixed seed, so the draw is reproducible).
sample = (frame.groupby('account_status', group_keys=False)
               .sample(n=N_PER_GROUP, random_state=SEED)
               .reset_index(drop=True))
assert sample['username_key'].is_unique
print(sample['account_status'].value_counts().to_string())

## 3. Attach Arctic Shift metadata and PRAW profiles

In [ ]:
# Attach Arctic Shift account metadata (prefix arctic_) and PRAW profile fields (prefix praw_) to the sample.
# Image URL columns are dropped; account_status comes from the sample, not the profile table.
meta = pd.read_parquet(os.path.join(DATA_DIR, 'account_meta_arctic.parquet'))
profiles = pd.read_parquet(os.path.join(DATA_DIR, 'account_profiles_full.parquet'))

meta = meta.drop(columns=['fetch_error']).add_prefix('arctic_').rename(columns={'arctic_username_key': 'username_key'})
profiles = (profiles.drop(columns=['account_status', 'icon_img', 'snoovatar_img'])
                    .add_prefix('praw_').rename(columns={'praw_username_key': 'username_key'}))

sample_accounts = (sample.merge(meta, on='username_key', how='left', validate='one_to_one')
                         .merge(profiles, on='username_key', how='left', validate='one_to_one'))
# Rows and columns of the joined table, then activity counts by status.
print(sample_accounts.shape)
sample_accounts.groupby('account_status')[['n_posts', 'n_comments', 'times_mentioned']].describe().T

## 4. Pull the sample's stored history
The `repair-*` files re-pulled whole histories for accounts whose pagination cursor was at risk,
so the same `id` can appear in both a `part-*` and a `repair-*` file. Rows are deduplicated on `id`.

In [ ]:
# Load the stored post and comment histories for the 200 sampled accounts only (outer_key = account).
keys = sample['username_key'].tolist()

def load_history(kind):
    # read file by file: some parts store all-null columns as type null, which breaks a unified dataset schema
    files = sorted(glob.glob(os.path.join(DATA_DIR, f'arctic_{kind}', '*.parquet')))
    parts = [pd.read_parquet(f, engine='pyarrow', filters=[('outer_key', 'in', keys)]) for f in files]
    df = pd.concat([p for p in parts if len(p)], ignore_index=True)
    n_raw = len(df)
    df = df.drop_duplicates('id').reset_index(drop=True)
    print(f'{kind}: {n_raw:,} rows -> {len(df):,} after dedupe on id, {df["outer_key"].nunique()} accounts')
    return df

# Posts first, then comments.
sample_posts = load_history('posts')
sample_comments = load_history('comments')

In [ ]:
# Rows stored per account, by status: a check that every sampled account actually has history.
status = sample.set_index('username_key')['account_status']
counts = pd.DataFrame({'posts': sample_posts['outer_key'].value_counts(),
                       'comments': sample_comments['outer_key'].value_counts()}).reindex(keys).fillna(0).astype(int)
counts['account_status'] = counts.index.map(status)
print('accounts with no stored rows:', ((counts['posts'] + counts['comments']) == 0).sum())
counts.groupby('account_status')[['posts', 'comments']].agg(['sum', 'median', 'mean', 'max'])

## 5. Save

In [ ]:
# never overwrite: the sampling frame grows as the collection gap run fills in, so a re-run would draw a different sample
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

# Account table, then the post and comment histories.
save_new(sample_accounts, 'sample_accounts.parquet')
save_new(sample_posts, 'sample_posts.parquet')
save_new(sample_comments, 'sample_comments.parquet')